In [ ]:
# import required modules 
    # copied these over from part 1
from abc_atlas_access.abc_atlas_cache.abc_project_cache import AbcProjectCache
from pathlib import Path
import anndata
import numpy as np
import os
import scanpy as sc
import pandas as pd
import time
import matplotlib.pyplot as plt

In [ ]:
# MOST RECENT DOWNSAMPLING METHOD (01/21/26)
# Rules: 
    # 1. downsample neuronal CLASSES randomly by 10%, 
        # 1a. if there are <1000 cells in the class, don't downsample
        # 1b. after downsampling, cap neuronal classes at 1000 cells, if needed 
    # 2. downsample nonneuronal CLUSTERS randomly by 10% 
        # 2a. unless the cluster has <1000 cells

# Settings:
DATA_DIR = "/Users/cclu223/Desktop/ABC_reference/WMB_10xv3_data/raw" # <-- CHANGE THIS
MIN_CELLS = 1000        
DOWNSAMPLE_RATE = 0.10  
MAX_NEURONAL = 1000     # neuronal cap
OUTPUT_FILE = "20260121_WMB_10xv3_downsampled.h5ad" # <-- CHANGE THIS

# column that defines cluster/class in metadata
CLUSTER_COL = "cluster"
CLASS_COL = "class"



# path to metadata with cluster annotations
METADATA_FILE =  "/Users/cclu223/Desktop/ABC_reference/WMB_10xv3_data/metadata/WMB-10X/20241115/views/cell_metadata_with_cluster_annotation.csv"


# load metadata
cell_meta = pd.read_csv(METADATA_FILE, index_col=0)


# find all .h5ad files in DATA_DIR
files = [os.path.join(DATA_DIR, f) for f in os.listdir(DATA_DIR) if f.endswith(".h5ad")]


# intialize list where the downsampled cells will be stored during for loop
subset_list = []


# loop through each file, and apply rules 
for f in files:
    
    print(f"\nProcessing: {os.path.basename(f)}")
    
    adata = sc.read_h5ad(f, backed="r")
    
    # create new metadata columns in the 10xv3 object for cluster information
    merged_obs = adata.obs.merge(cell_meta[[CLUSTER_COL, CLASS_COL]], 
                                 left_on=adata.obs_names, 
                                 right_index=True, 
                                 how='left')
    
    # add metadata values in the new columns in the 10xv3 obj
    adata.obs[CLUSTER_COL] = merged_obs[CLUSTER_COL].values
    
    adata.obs[CLASS_COL] = merged_obs[CLASS_COL].values

    # intialize keep_cells list
    keep_cells = []

    np.random.seed(42)

    # loop over classes (highest annotation)
    for class_name, class_df in adata.obs.groupby(CLASS_COL):

        # if there is no class name, keep the cells and move on to next block 
            # prevents errors in rest of loop, and these unannotated cells will be removed later on 
        if pd.isna(class_name):
            
            keep_cells.extend(class_df.index.tolist())
            
            continue

        # classifying what is considered a neuronal class (if the class name has Glut, GABA, Sero, or Dopa, it is a neuronal cluster)
        IS_NEURONAL = bool(pd.Series(class_name).str.contains(r"Glut|GABA|Dopa|Sero", regex = True, na = False).iloc[0])

        
        # number of cells in the class 
        n_class = class_df.shape[0]

        # downsampling neurons by class only 
        if IS_NEURONAL:

            # RULE: if there are <1000 in the class, don't downsample 
            if n_class <= MIN_CELLS:
                
                keep_cells.extend(class_df.index.tolist())
                
                continue

            # number of cells to keep after downsampling 
            n_keep = max(1, int(n_class * DOWNSAMPLE_RATE))

            # randomly downsampling that number of cells
            sampled = np.random.choice(class_df.index, size=n_keep, replace=False)

            # RULE: if neuronal class has >1000 cells after downsampling, cap it at 1000 cells 
            if len(sampled) > MAX_NEURONAL:
                
                sampled = np.random.choice(sampled, size=MAX_NEURONAL, replace=False)
                
                print(f"  Neuronal cap: {class_name} → {MAX_NEURONAL}")

            # RULE: if not, add downsampled neuronal cells to keep_cells list
            else:
                print(f"  Neuronal downsample: {class_name} → {len(sampled)}")

            keep_cells.extend(sampled)
            
            continue

        # split by cluster for nonneuronal cell types 
        for cluster_name, cluster_df in class_df.groupby(CLUSTER_COL):

            # number of cells in cluster
            n_cluster = cluster_df.shape[0]

            # RULE: if cluster has <1000 cells, don't downsample 
            if n_cluster < MIN_CELLS:
                
                keep_cells.extend(cluster_df.index.tolist())
                
                continue

            #RULE: downsample all other nonneuronal clusters by 10%
            n_keep = max(1, int(n_cluster * DOWNSAMPLE_RATE))
            
            sampled = np.random.choice(cluster_df.index, size = n_keep, replace = False)

            # add these downsampled cells to keep_cells list
            keep_cells.extend(sampled)

    # Load fully in memory only selected cells
    adata_sub = sc.read_h5ad(f)[keep_cells, :].copy()

    # add keep_cells to subset_list after looping through each file 
    subset_list.append(adata_sub)

# merge all downsampled datasets 
merged = sc.concat(subset_list, join="outer", label="dataset", keys=[os.path.basename(f) for f in files])

print(f"\nFinal dataset cells: {merged.n_obs:,}, genes: {merged.n_vars:,}")

# save merged and downsampled file 
merged.write_h5ad(OUTPUT_FILE)

print(f"\nSaved merged downsampled dataset to: {OUTPUT_FILE}")

# REPEAT WITH LOG2 FILES!



In [ ]:
# checking to see which cells and clusters were preserved with downsampling 

# loading merged and downsampled h5ad object and metadata file 

adata = anndata.read_h5ad("/Users/cclu223/Desktop/ABC_reference/WMB_10xv3_data/downsampled_objs/20260121_WMB_10xv3_downsampled.h5ad")
adata

# load in cell cluster metadata file 
cell = pd.read_csv("/Users/cclu223/Desktop/ABC_reference/WMB_10xv3_data/metadata/WMB-10X/20241115/views/cell_metadata_with_cluster_annotation.csv")

cell.set_index('cell_label', inplace=True)

print("Number of cells = ", len(cell))

# look at first 5 cells
print(cell.head(5))

# sanity check 

adata.obs_names[:5]
cell.index[:5]

# creating df with downsampled cell counts
cell_ds = cell.loc[cell.index.isin(adata.obs_names)]


# sanity check: making sure number of downsampled cells and number of observations in the object match 

len(cell_ds), adata.n_obs # outputs should match 

# attach cluster annotations to merged and downsampled object 

adata.obs = adata.obs.join(
    cell[['class', 'subclass', 'supertype', 'cluster']],
    how='left'
)

# print how many cells are in each class in the downsampled and merged object 
adata.obs['class'].value_counts()

# save downsampled object with cluster metadata added in
adata.write_h5ad("/Users/cclu223/Desktop/ABC_reference/WMB_10xv3_data/downsampled_objs/20260121_WMB_10xv3_downsampled_and_metadata.h5ad")

In [ ]:
# comparing how much each cluster was downsampled 

# cell = original cell_metadata_with_cluster_annotation file (not downsampled) 
# cell_ds = corresponding downsampled version of cell_metadata_with_cluster_annotation

# per class (most general clustering) 
og_cells_per_class = (
    cell
    .groupby('class')
    .size()
    .reset_index(name='cell_count')
    .sort_values('cell_count', ascending=False)
)

ds_cells_per_class = (
    cell_ds
    .groupby('class')
    .size()
    .reset_index(name='cell_count')
    .sort_values('cell_count', ascending=False)
)

# combining dataframes to visualize differences

combined = og_cells_per_class.merge(
    ds_cells_per_class,
    on='class',
    suffixes=('_original', '_downsampled')
)

combined



In [ ]:
# per subclass 

og_cells_per_subclass = (
    cell
    .groupby('subclass')
    .size()
    .reset_index(name='cell_count')
    .sort_values('cell_count', ascending=False)
)

ds_cells_per_subclass = (
    cell_ds
    .groupby('subclass')
    .size()
    .reset_index(name='cell_count')
    .sort_values('cell_count', ascending=False)
)

combined1 = og_cells_per_subclass.merge(
    ds_cells_per_subclass,
    on='subclass',
    suffixes=('_original', '_downsampled')
)

combined1[:-50]

In [ ]:
# filtering by NN cell types to see if they are conserved in cluster hierarchy

# Original dataset
og_cells_per_subclass_NN = og_cells_per_subclass[og_cells_per_subclass['subclass'].str.endswith('NN')]

# Downsampled dataset
ds_cells_per_subclass_NN = ds_cells_per_subclass[ds_cells_per_subclass['subclass'].str.endswith('NN')]

combined_NN = og_cells_per_subclass_NN.merge(
    ds_cells_per_subclass_NN,
    on='subclass',
    suffixes=('_original', '_downsampled')
)

combined_NN


In [ ]:
# per supertype

cells_per_supertype = (
    cell
    .groupby('supertype')
    .size()
    .reset_index(name='cell_count')
    .sort_values('cell_count', ascending=False)
)

cells_per_supertype

# per cluster (most detailed clustering)

cells_per_cluster = (
    cell
    .groupby('cluster')
    .size()
    .reset_index(name='cell_count')
    .sort_values('cell_count', ascending=False)
)

cells_per_cluster